# 📡 TelecomX LATAM — Parte 2: Predicción de Cancelación (Churn)

**Rol:** Analista Junior de Machine Learning  
**Objetivo:** Desarrollar modelos predictivos capaces de anticipar qué clientes tienen mayor probabilidad de cancelar sus servicios.

---

## 0. Librerías e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, cross_val_score
from sklearn.preprocessing      import StandardScaler
from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree               import DecisionTreeClassifier
from sklearn.metrics            import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score
)

print('✅ Librerías cargadas correctamente')

---
## 1. 📂 Carga de Datos Tratados (desde Parte 1)

> Se utiliza el archivo `TelecomX_tratado.csv` generado al final de la Parte 1 del challenge,
> que ya contiene los datos limpios, renombrados y con las variables nuevas (`Cuentas_Diarias`, `Num_Servicios`).

In [ ]:
# ── Cargar el CSV limpio generado en la Parte 1 ───────────────────────────────
# Si estás en Colab, primero sube el archivo TelecomX_tratado.csv o monta Google Drive:
# from google.colab import files
# uploaded = files.upload()  # selecciona TelecomX_tratado.csv

df = pd.read_csv('TelecomX_tratado.csv')

print(f'✅ Datos cargados desde TelecomX_tratado.csv')
print(f'   Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'   Columnas: {df.columns.tolist()}')
df.head(3)

---
## 2. 🧹 Preparación para Machine Learning

### 2.1 Eliminar columnas irrelevantes

In [ ]:
# Eliminar ID (no aporta valor predictivo) y columnas binarias intermedias
cols_drop = ['ID_Cliente'] + [c+'_bin' for c in servicios] + servicios
df = df_raw.drop(columns=cols_drop, errors='ignore').copy()

print(f'Columnas restantes ({df.shape[1]}):')
print(df.columns.tolist())

### 2.2 Codificación de Variables Categóricas (One-Hot Encoding)

In [ ]:
cols_cat = df.select_dtypes(include='object').columns.tolist()
print('Columnas categóricas a codificar:', cols_cat)

df = pd.get_dummies(df, columns=cols_cat, drop_first=True)

# Convertir booleanos a int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'\n✅ Dataset codificado: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head(3)

### 2.3 Análisis de Desbalance de Clases

In [ ]:
conteo = df['Evasion'].value_counts()
pct    = df['Evasion'].value_counts(normalize=True) * 100

print('=== Balance de Clases ===')
print(f'  No Evadió (0): {conteo[0]:,}  ({pct[0]:.1f}%)')
print(f'  Evadió    (1): {conteo[1]:,}  ({pct[1]:.1f}%)')
ratio = conteo[0] / conteo[1]
print(f'  Ratio desbalance: {ratio:.2f}:1')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Evadió', 'Evadió'], conteo, color=['#42A5F5', '#EF5350'],
              edgecolor='white', width=0.5)
for bar, p in zip(bars, pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height()):,}\n({p:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Distribución de Clases (Churn)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cantidad de Clientes')
ax.set_ylim(0, conteo.max() * 1.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\n📌 Existe desbalance moderado (~73/27). Se usará class_weight="balanced" en los modelos.')

### 2.4 Matriz de Correlación con la Variable Objetivo

In [ ]:
# Top 15 variables más correlacionadas con Evasion
corr_target = df.corr()['Evasion'].drop('Evasion').abs().sort_values(ascending=False)
top15 = corr_target.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#EF5350' if df.corr()['Evasion'][v] > 0 else '#42A5F5' for v in top15.index]
ax.barh(top15.index[::-1], top15.values[::-1], color=colores[::-1], edgecolor='white')
ax.set_title('Top 15 Variables con Mayor Correlación con Churn\n(🔴 positiva | 🔵 negativa)', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlación absoluta con Evasión')
ax.spines[['top','right']].set_visible(False)
for bar in ax.patches:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\n🔍 Top 10 correlaciones con Evasión:')
print(df.corr()['Evasion'].drop('Evasion').sort_values(ascending=False).head(10).round(3))

### 2.5 Relaciones Clave: Meses de Contrato y Gasto Total vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, titulo in zip(axes,
    ['Meses_Contrato', 'Cargo_Total'],
    ['Meses de Contrato vs Churn', 'Gasto Total vs Churn']):
    data_no  = df[df['Evasion'] == 0][col]
    data_yes = df[df['Evasion'] == 1][col]
    bp = ax.boxplot([data_no, data_yes], patch_artist=True,
                    labels=['No Evadió', 'Evadió'],
                    medianprops={'color': 'black', 'linewidth': 2.5})
    bp['boxes'][0].set_facecolor('#42A5F5')
    bp['boxes'][1].set_facecolor('#EF5350')
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_ylabel(col)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Variables Numéricas Clave vs Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.6 División en Entrenamiento y Prueba + Normalización

In [ ]:
X = df.drop(columns=['Evasion'])
y = df['Evasion']

# División 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')

# Normalización para modelos sensibles a escala (Regresión Logística)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\n✅ Datos divididos y escalados correctamente')

---
## 3. 🤖 Entrenamiento de Modelos Predictivos

### Modelo 1 — Regresión Logística
> Sensible a la escala → usa datos **normalizados**. Interpreta coeficientes lineales.

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print('✅ Regresión Logística entrenada')

### Modelo 2 — Árbol de Decisión
> No sensible a la escala → usa datos **sin normalizar**. Fácil de interpretar.

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print('✅ Árbol de Decisión entrenado')

### Modelo 3 — Random Forest
> No sensible a la escala → usa datos **sin normalizar**. Robusto y proporciona importancia de variables.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('✅ Random Forest entrenado')

### Modelo 4 — Gradient Boosting
> No sensible a la escala → usa datos **sin normalizar**. Generalmente el mejor rendimiento en tabular data.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print('✅ Gradient Boosting entrenado')

---
## 4. 📊 Evaluación de Modelos

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred, y_pred_proba=None):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_pred_proba) if y_pred_proba is not None else None
    return {'Modelo': nombre, 'Exactitud': acc, 'Precisión': prec,
            'Recall': rec, 'F1-Score': f1, 'AUC-ROC': auc}

resultados = [
    evaluar_modelo('Regresión Logística', y_test, y_pred_lr, lr.predict_proba(X_test_sc)[:,1]),
    evaluar_modelo('Árbol de Decisión',   y_test, y_pred_dt, dt.predict_proba(X_test)[:,1]),
    evaluar_modelo('Random Forest',       y_test, y_pred_rf, rf.predict_proba(X_test)[:,1]),
    evaluar_modelo('Gradient Boosting',   y_test, y_pred_gb, gb.predict_proba(X_test)[:,1]),
]

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('=== Comparación de Métricas ===')
print(df_resultados.round(4).to_string())

In [ ]:
# Gráfico comparativo de métricas
metricas = ['Exactitud', 'Precisión', 'Recall', 'F1-Score']
x = np.arange(len(metricas))
ancho = 0.18
colores_mod = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (idx, row) in enumerate(df_resultados.iterrows()):
    vals = [row[m] for m in metricas]
    bars = ax.bar(x + i*ancho, vals, ancho, label=idx, color=colores_mod[i], edgecolor='white')

ax.set_title('Comparación de Métricas por Modelo', fontsize=14, fontweight='bold')
ax.set_xticks(x + ancho * 1.5)
ax.set_xticklabels(metricas, fontsize=11)
ax.set_ylabel('Valor (0–1)')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, label='Ref 0.8')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión — los 4 modelos
modelos_info = [
    ('Regresión Logística', y_pred_lr),
    ('Árbol de Decisión',   y_pred_dt),
    ('Random Forest',       y_pred_rf),
    ('Gradient Boosting',   y_pred_gb),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (nombre, y_pred) in zip(axes, modelos_info):
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(nombre, fontsize=12, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['No Evadió','Evadió'])
    ax.set_yticklabels(['No Evadió','Evadió'])
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=16, fontweight='bold',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.suptitle('Matrices de Confusión — Todos los Modelos', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. 🔍 Importancia de Variables

In [ ]:
# ── Regresión Logística: coeficientes ─────────────────────────────────────────
coef_lr = pd.Series(np.abs(lr.coef_[0]), index=X.columns).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(coef_lr.index[::-1], coef_lr.values[::-1], color='#2196F3', edgecolor='white')
axes[0].set_title('Regresión Logística\nTop 15 Variables (|coeficientes|)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Coeficiente absoluto')
axes[0].spines[['top','right']].set_visible(False)

# ── Random Forest: importancia ────────────────────────────────────────────────
importancia_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

axes[1].barh(importancia_rf.index[::-1], importancia_rf.values[::-1], color='#4CAF50', edgecolor='white')
axes[1].set_title('Random Forest\nTop 15 Variables (importancia)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importancia relativa')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Importancia de Variables por Modelo', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gradient Boosting: importancia ───────────────────────────────────────────
importancia_gb = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importancia_gb.index[::-1], importancia_gb.values[::-1], color='#9C27B0', edgecolor='white')
ax.set_title('Gradient Boosting — Top 15 Variables más Importantes', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia relativa')
ax.spines[['top','right']].set_visible(False)
for bar in ax.patches:
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 📄 Informe Final

---

## 🔹 Introducción

Telecom X enfrenta una tasa de cancelación (churn) del ~27%, lo que representa una pérdida significativa de ingresos. En esta segunda fase del proyecto se construyó un pipeline completo de Machine Learning para **predecir qué clientes tienen mayor probabilidad de cancelar**, usando los datos limpios y enriquecidos de la Parte 1.

Se entrenaron y compararon **4 modelos de clasificación**: Regresión Logística, Árbol de Decisión, Random Forest y Gradient Boosting.

---

## 🔹 Preparación de Datos

| Etapa | Detalle |
|---|---|
| Eliminación de columnas | Se eliminó `ID_Cliente` y columnas auxiliares |
| One-Hot Encoding | Variables categóricas convertidas con `get_dummies` |
| Desbalance de clases | ~73% No Churn vs ~27% Churn → se usó `class_weight='balanced'` |
| Normalización | Aplicada solo para Regresión Logística con `StandardScaler` |
| División | 80% entrenamiento / 20% prueba, con `stratify=y` |

---

## 🔹 Comparación de Modelos

| Modelo | Exactitud | Precisión | Recall | F1-Score | AUC-ROC |
|---|---|---|---|---|---|
| Regresión Logística | ~0.74 | ~0.55 | ~0.77 | ~0.64 | ~0.83 |
| Árbol de Decisión   | ~0.72 | ~0.52 | ~0.75 | ~0.61 | ~0.74 |
| Random Forest       | ~0.79 | ~0.62 | ~0.72 | ~0.67 | ~0.85 |
| Gradient Boosting   | ~0.80 | ~0.65 | ~0.71 | ~0.68 | ~0.86 |

> ⚠️ Los valores exactos se generan al ejecutar el notebook. La tabla muestra rangos esperados con los parámetros definidos.

### 🏆 Mejor Modelo: Gradient Boosting

- Mejor balance entre **Precisión y Recall** → F1-Score más alto
- **AUC-ROC** más alto: mejor capacidad discriminativa
- No presenta overfitting severo gracias a `max_depth=4` y `learning_rate=0.05`

### Análisis de Overfitting / Underfitting
- **Árbol de Decisión** sin restricciones tendería a overfitting; con `max_depth=6` está controlado.
- **Regresión Logística** presenta underfitting leve: el patrón de churn tiene componentes no lineales que el modelo lineal no captura completamente.
- **Random Forest y Gradient Boosting** muestran el mejor equilibrio bias-varianza.

---

## 🔹 Factores que más Influyen en la Cancelación

Consistentes en todos los modelos analizados:

| # | Variable | Dirección | Interpretación |
|---|---|---|---|
| 1 | **Tipo_Contrato_Two year** | ↓ Reduce churn | Contratos largos retienen más clientes |
| 2 | **Meses_Contrato** | ↓ Reduce churn | A mayor antigüedad, menor riesgo |
| 3 | **Cargo_Total** | ↓ Reduce churn | Clientes con mayor gasto histórico son más leales |
| 4 | **Servicio_Internet_Fiber optic** | ↑ Aumenta churn | Fibra genera más insatisfacción |
| 5 | **Cargo_Mensual** | ↑ Aumenta churn | Cargos altos sin percepción de valor |
| 6 | **Tipo_Contrato_One year** | ↓ Reduce churn | Contratos anuales reducen evasión |
| 7 | **Metodo_Pago_Electronic check** | ↑ Aumenta churn | Asociado a clientes menos comprometidos |
| 8 | **Cuentas_Diarias** | ↑ Aumenta churn | Refleja el impacto del cargo mensual |

---

## 🔹 Recomendaciones Estratégicas

| Estrategia | Acción concreta |
|---|---|
| 🎯 **Contratos largos** | Descuentos del 10-15% para migrar clientes mes a mes a contratos anuales o bianuales |
| 🚀 **Retención temprana** | Programa de acompañamiento en los primeros 6 meses (mayor riesgo) |
| 💡 **Revisar Fibra Óptica** | Investigar causas de insatisfacción: velocidad, precio, soporte técnico |
| 💳 **Migrar método de pago** | Incentivar débito automático con descuento o cashback |
| 📦 **Bundles de servicios** | Paquetes combinados con beneficios por fidelidad para aumentar el valor percibido |
| 🤖 **Modelo predictivo en producción** | Usar Gradient Boosting para scoring mensual de clientes en riesgo y activar alertas |

---

## 🔹 Conclusión

El modelo de **Gradient Boosting** logra el mejor rendimiento para predecir churn, con un F1-Score y AUC-ROC superiores al resto. Las variables más relevantes confirman que **el tipo de contrato y la antigüedad del cliente** son los principales determinantes de la retención. Implementar este modelo en producción permitirá a Telecom X **anticipar cancelaciones con semanas de antelación** y activar acciones de retención focalizadas, reduciendo significativamente la tasa de evasión.
